# 11-2. 흥행 기간 변수 추가 (KOBIS 일별 데이터 재활용)

11 잔차 분석 결론: 변호인(과소 예측)과 수상한 그녀(과대 예측)는 **개봉 스크린 수 · 상위 10위 일수가 비슷한데 결과가 반대** → 지금 변수로는 구분 불가.
→ "비슷하게 시작했지만 다르게 끝난" 영화를 구분하려고 **상영 흐름(시간에 따른 스크린 · 상영 횟수)** 변수를 추가.

이미 받아 둔 `kobis_cache/` (일별 상위 10위 · 스크린 수 · 상영 횟수 · 순위)로 계산 → **API 추가 호출 없음**

| 변수 | 설명 |
| --- | --- |
| `t_scrn_d7` · `t_scrn_d14` · `t_scrn_d21` | 개봉 7 · 14 · 21일째 스크린 수 (그날 10위 밖이면 0) |
| `t_show_d7` · `t_show_d14` | 개봉 7 · 14일째 상영 횟수 |
| `t_keep_d7` · `t_keep_d14` | 7 · 14일째 스크린 ÷ 개봉일 스크린 (흥행 유지율) |
| `t_sum_scrn` · `t_sum_show` | 상위 10위 기간 스크린 수 합 · 상영 횟수 합 (상영 규모 × 기간) |
| `t_days_rank1` | 1위 일수 |
| `t_last_day` | 개봉 후 마지막으로 10위 안에 든 날 |

⚠️ 상영이 오래 이어진 건 관객이 많았기 때문 → **정답 정보가 간접적으로 들어가는 정도가 09보다 강함**. 발표 때 밝히기.

| 실험 | 변수 |
| --- | --- |
| T0 | 09 그대로 (비교 기준, CV 363,107 · Public 436,672) |
| T1 | 09 + 흥행 기간 변수 전체 |
| T2 | 09 + 스크린 · 상영 횟수 흐름만 (순위 · 1위 일수 · 마지막 날 같은 **순위 기반 변수는 제외**) |

## 0. 설정

In [1]:
import os
import re
import glob
import json
import time
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

from features import make_features

warnings.filterwarnings('ignore')
train = pd.read_csv('movies_train.csv')
test = pd.read_csv('movies_test.csv')
submission = pd.read_csv('submission.csv')
kobis = pd.read_csv('kobis_movies.csv', dtype={'movieCd': str})
y = train['box_off_num'].values.astype(float)
hit = y >= 1_000_000
rmse = lambda a, b: mean_squared_error(a, b) ** 0.5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)
CACHE_DIR = 'cache_11_2'
os.makedirs(CACHE_DIR, exist_ok=True)

C:\Users\taehe\OneDrive\문서\대학교\3학년 2학기\ESAA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 일별 데이터 → 영화별 흥행 기간 변수

In [2]:
rows = []
for p in glob.glob('kobis_cache/*.json'):
    d = os.path.basename(p)[:8]
    for r in json.load(open(p, encoding='utf-8')):
        rows.append({**r, 'targetDt': pd.Timestamp(d)})
daily = pd.DataFrame(rows)
daily['movieCd'] = daily['movieCd'].astype(str)
for c in ['rank', 'scrnCnt', 'showCnt']:
    daily[c] = pd.to_numeric(daily[c], errors='coerce')
daily['open_dt'] = pd.to_datetime(daily['openDt'].str.strip().replace('', None), errors='coerce')
daily['day'] = (daily['targetDt'] - daily['open_dt']).dt.days        # 개봉 후 며칠째
print('일별 기록', len(daily), '행 · 영화', daily['movieCd'].nunique(), '편')


def traj(g):
    g = g[g['day'] >= 0]
    on = lambda n, col: g.loc[g['day'] == n, col].max() if (g['day'] == n).any() else 0.0
    open_scrn = g.loc[g['day'] == g['day'].min(), 'scrnCnt'].max() if len(g) else np.nan
    out = {'t_scrn_d7': on(7, 'scrnCnt'), 't_scrn_d14': on(14, 'scrnCnt'), 't_scrn_d21': on(21, 'scrnCnt'),
           't_show_d7': on(7, 'showCnt'), 't_show_d14': on(14, 'showCnt'),
           't_sum_scrn': g['scrnCnt'].sum(), 't_sum_show': g['showCnt'].sum(),
           't_days_rank1': (g['rank'] == 1).sum(), 't_last_day': g['day'].max() if len(g) else 0}
    out['t_keep_d7'] = out['t_scrn_d7'] / open_scrn if open_scrn else 0.0
    out['t_keep_d14'] = out['t_scrn_d14'] / open_scrn if open_scrn else 0.0
    return pd.Series(out)


T = daily.groupby('movieCd').apply(traj).fillna(0)
T_COLS = list(T.columns)
display(T.describe().T[['mean', '50%', 'max']].style.format('{:,.1f}'))

일별 기록 22510 행 · 영화 1371 편


,mean,50%,max
t_scrn_d7,199.4,136.0,"1,785.0"
t_scrn_d14,120.4,0.0,"1,307.0"
t_scrn_d21,71.0,0.0,930.0
t_show_d7,802.2,294.0,"9,153.0"
t_show_d14,468.9,0.0,"6,146.0"
t_sum_scrn,"5,038.9","2,517.0","50,506.0"
t_sum_show,"20,010.7","7,830.0","210,028.0"
t_days_rank1,1.6,0.0,37.0
t_last_day,73.7,13.0,"7,295.0"
t_keep_d7,0.5,0.6,3.6


## 2. train · test에 매칭 (09와 같은 기준: 제목 + 개봉일 30일 이내)

In [3]:
norm = lambda s: re.sub(r'[^0-9a-zA-Z가-힣]', '', str(s)).lower()
kobis['key'] = kobis['movieNm'].map(norm)
kobis['open_dt'] = pd.to_datetime(kobis['openDt'], errors='coerce')
K_COLS = ['open_scrn', 'open_show', 'max_scrn', 'max_show', 'days_top10', 'best_rank']


def kobis_features(df):
    out = []
    for title, r in zip(df['title'], pd.to_datetime(df['release_time'])):
        cand = kobis[kobis['key'] == norm(title)]
        row = {f'k_{c}': np.nan for c in K_COLS} | {c: np.nan for c in T_COLS}
        if len(cand):
            gap = (cand['open_dt'] - r).abs().dt.days.fillna(9999)
            if gap.min() <= 30:
                best = cand.loc[gap.idxmin()]
                row = {f'k_{c}': best[c] for c in K_COLS} | (T.loc[best['movieCd']].to_dict() if best['movieCd'] in T.index else {})
        out.append(row)
    K = pd.DataFrame(out)
    K['k_in_top10'] = K['k_max_scrn'].notna().astype(int)
    K['k_best_rank'] = K['k_best_rank'].fillna(11)
    return K.fillna(0).astype(float)


base_tr, base_te = make_features(train, test, {'debut': 1, 'debut_mode': 'nan', 'distributor': 1, 'date': 1, 'rating': 1})
KT_tr, KT_te = kobis_features(train), kobis_features(test)
print('흥행 기간 변수가 있는 영화  train', int((KT_tr['t_sum_scrn'] > 0).sum()), '/ test', int((KT_te['t_sum_scrn'] > 0).sum()))

# 11에서 반대로 틀린 두 영화: 새 변수로 구분되는지
pair = train['title'].isin(['변호인', '수상한 그녀'])
display(pd.concat([train.loc[pair, ['title', 'box_off_num']].reset_index(drop=True),
                   KT_tr.loc[pair.values, ['k_open_scrn', 'k_days_top10', 't_scrn_d14', 't_keep_d14', 't_sum_show', 't_last_day']].reset_index(drop=True)], axis=1)
        .style.format({c: '{:,.2f}' for c in ['t_keep_d14']} | {c: '{:,.0f}' for c in ['box_off_num', 'k_open_scrn', 'k_days_top10', 't_scrn_d14', 't_sum_show', 't_last_day']}))

hit_mask = (KT_tr['k_in_top10'] == 1).values
rel = KT_tr[T_COLS].loc[hit_mask].corrwith(pd.Series(y[hit_mask], index=KT_tr.index[hit_mask]), method='spearman')
display(rel.sort_values(ascending=False).to_frame('관객수와 Spearman (10위 진입 영화)').style.format('{:.3f}'))

흥행 기간 변수가 있는 영화  train 269 / test 109


,title,box_off_num,k_open_scrn,k_days_top10,t_scrn_d14,t_keep_d14,t_sum_show,t_last_day
0,변호인,"11,374,879",591,60,887,1.50,"150,947",61
1,수상한 그녀,"8,659,725",577,71,773,1.34,"136,524",70


,관객수와 Spearman (10위 진입 영화)
t_sum_scrn,0.981
t_sum_show,0.981
t_show_d7,0.930
t_scrn_d7,0.920
t_show_d14,0.915
t_scrn_d14,0.914
t_scrn_d21,0.846
t_last_day,0.828
t_keep_d14,0.825
t_keep_d7,0.745


## 3. TabPFN CV (09와 같은 설정: v3.5 · sqrt 타깃)

In [4]:
RANK_BASED = ['t_days_rank1', 't_last_day']
SETS = {
    'T0_09기준': [c for c in KT_tr.columns if not c.startswith('t_')],
    'T1_흐름전체': list(KT_tr.columns),
    'T2_흐름_순위제외': [c for c in KT_tr.columns if c not in RANK_BASED],
}


def run(name):
    path = os.path.join(CACHE_DIR, f'{name}.pkl')
    if os.path.exists(path):
        return pickle.load(open(path, 'rb'))
    cols = SETS[name]
    X = pd.concat([base_tr.reset_index(drop=True), KT_tr[cols]], axis=1)
    Xt = pd.concat([base_te.reset_index(drop=True), KT_te[cols]], axis=1)
    mk = lambda: TabPFNRegressor.create_default_for_version(ModelVersion.V3_5, device=DEVICE, random_state=42)
    oof, fold = np.zeros(len(y)), []
    t0 = time.time()
    for tr_idx, va_idx in kf5.split(X):
        m = mk(); m.fit(X.iloc[tr_idx], np.sqrt(y[tr_idx]))
        oof[va_idx] = np.square(np.clip(m.predict(X.iloc[va_idx]), 0, None))
        fold.append(rmse(y[va_idx], oof[va_idx]))
    m = mk(); m.fit(X, np.sqrt(y))
    res = {'실험': name, '변수 수': X.shape[1], 'CV RMSE': rmse(y, oof), 'fold 표준편차': np.std(fold),
           '흥행작 RMSE': rmse(y[hit], oof[hit]), '소요(초)': time.time() - t0,
           'oof': oof, 'test': np.square(np.clip(m.predict(Xt), 0, None))}
    pickle.dump(res, open(path, 'wb'))
    return res


# T0는 09와 같은 조건 → 09 결과 재사용 (같은 값이 나오는지도 확인 가능)
if not os.path.exists(os.path.join(CACHE_DIR, 'T0_09기준.pkl')) and os.path.exists('cache_09/tabpfn.pkl'):
    p9 = pickle.load(open('cache_09/tabpfn.pkl', 'rb'))
    pickle.dump({'실험': 'T0_09기준', '변수 수': base_tr.shape[1] + len(SETS['T0_09기준']), 'CV RMSE': rmse(y, p9['oof']),
                 'fold 표준편차': np.nan, '흥행작 RMSE': rmse(y[hit], p9['oof'][hit]), '소요(초)': 0,
                 'oof': p9['oof'], 'test': p9['test']}, open(os.path.join(CACHE_DIR, 'T0_09기준.pkl'), 'wb'))

R = {}
for name in SETS:
    R[name] = r = run(name)
    print(f"{name:<14} 변수 {r['변수 수']:>2} | CV RMSE {r['CV RMSE']:>9,.0f} | 흥행작 {r['흥행작 RMSE']:>9,.0f} | {r['소요(초)']:.0f}초")

T0_09기준        변수 48 | CV RMSE   363,107 | 흥행작   837,828 | 0초


T1_흐름전체        변수 59 | CV RMSE   292,008 | 흥행작   668,626 | 71초


T2_흐름_순위제외     변수 57 | CV RMSE   290,485 | 흥행작   664,424 | 64초


In [5]:
tbl = pd.DataFrame([{k: v for k, v in r.items() if k not in ('oof', 'test')} for r in R.values()])
tbl['09 대비'] = tbl['CV RMSE'] - R['T0_09기준']['CV RMSE']
display(tbl.style.format({'CV RMSE': '{:,.0f}', 'fold 표준편차': '{:,.0f}', '흥행작 RMSE': '{:,.0f}', '소요(초)': '{:.0f}', '09 대비': '{:+,.0f}'})
        .highlight_min(subset=['CV RMSE', '흥행작 RMSE'], color='#c6efce'))

# 1단계 오차가 컸던 영화가 좋아졌는지
p0 = R['T0_09기준']['oof']
idx = np.argsort(-(y - p0) ** 2)[:10]
cmp = pd.DataFrame({'제목': train['title'].values[idx], '실제': y[idx], '09': p0[idx],
                    'T1': R['T1_흐름전체']['oof'][idx], 'T2': R['T2_흐름_순위제외']['oof'][idx]})
display(cmp.style.format({c: '{:,.0f}' for c in ['실제', '09', 'T1', 'T2']}))

,실험,변수 수,CV RMSE,fold 표준편차,흥행작 RMSE,소요(초),09 대비
0,T0_09기준,48,"363,107",nan,"837,828",0,+0
1,T1_흐름전체,59,"292,008","44,777","668,626",71,"-71,100"
2,T2_흐름_순위제외,57,"290,485","47,172","664,424",64,"-72,622"


,제목,실제,09,T1,T2
0,변호인,"11,374,879","8,495,714","10,898,771","10,682,976"
1,수상한 그녀,"8,659,725","11,436,890","9,995,731","10,225,589"
2,타워,"5,181,014","3,023,113","4,140,294","4,051,270"
3,국제시장,"14,262,766","12,194,612","14,148,516","14,388,543"
4,관상,"9,135,806","7,206,827","6,748,143","6,856,902"
5,7번방의 선물,"12,811,435","10,894,412","11,122,065","11,199,897"
6,감시자들,"5,508,299","7,421,600","6,580,907","6,612,923"
7,도둑들,"12,983,841","11,228,738","10,384,207","10,596,282"
8,"광해, 왕이 된 남자","12,323,595","14,066,666","14,011,642","14,184,996"
9,전설의 주먹,"1,744,585","3,467,440","2,826,151","2,834,972"


## 4. 제출 파일

In [6]:
for name in ['T1_흐름전체', 'T2_흐름_순위제외']:
    r = R[name]
    sub = submission.copy(); sub['box_off_num'] = r['test']
    fname = f'11_2_{name}.csv'
    sub.to_csv(fname, index=False)
    gain = R['T0_09기준']['CV RMSE'] - r['CV RMSE']
    print(f"{fname:<28} CV {r['CV RMSE']:,.0f} (09 대비 {-gain:+,.0f}, {gain/R['T0_09기준']['CV RMSE']*100:.1f}% 개선)")
tbl.to_csv('11_2_results.csv', index=False, encoding='utf-8-sig')

11_2_T1_흐름전체.csv             CV 292,008 (09 대비 -71,100, 19.6% 개선)
11_2_T2_흐름_순위제외.csv          CV 290,485 (09 대비 -72,622, 20.0% 개선)
